# Job Task: Update Serving Endpoint

> **This notebook runs as a Databricks Job task.**

Task 4 (final) of the retrain pipeline:
- Update the shared instructor reference endpoint to the new champion version
- Verify the endpoint becomes Ready

In [0]:
dbutils.widgets.text("catalog", "workshop", "Catalog")
dbutils.widgets.text("schema", "00_shared", "Schema")
dbutils.widgets.text("endpoint_name", "churn_instructor_reference", "Endpoint Name")

catalog       = dbutils.widgets.get("catalog")
schema        = dbutils.widgets.get("schema")
endpoint_name = dbutils.widgets.get("endpoint_name")

In [0]:
%pip install /Volumes/{catalog}/00_shared/wheels/churn_model-0.1.0-py3-none-any.whl --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedModelInput,
)

catalog       = dbutils.widgets.get("catalog")
schema        = dbutils.widgets.get("schema")
endpoint_name = dbutils.widgets.get("endpoint_name")

new_version = dbutils.jobs.taskValues.get(taskKey="gate_and_register", key="new_version")
model_name  = f"{catalog}.{schema}.churn_classifier"

w = WorkspaceClient()

print(f"Updating endpoint '{endpoint_name}' to model version {new_version}")

w.serving_endpoints.update_config_and_wait(
    name=endpoint_name,
    served_models=[
        ServedModelInput(
            model_name=model_name,
            model_version=new_version,
            workload_size="Small",
            scale_to_zero_enabled=True,
        )
    ],
)

print(f"✓ Endpoint updated to version {new_version}")